# 📅 2026-03-25 (수) 개발 노트 : 스팀 데이터 5,000개 추출 및 Hidden Gem 하이브리드 추천 엔진 완성

## 🎯 오늘의 목표
- [x] Steam API를 활용한 초기 게임 데이터 5,000개 추출 및 DB 구축 (`data_worker.py`)
- [x] 추출된 데이터에 대한 GPT-4o-mini 14개 지표(0~100점) 분석 및 임베딩 파이프라인 완성
- [x] 데이터 처리 로직과 추천 API 서버(`main.py`)의 완벽한 관심사 분리
- [x] Vector 검색(의미론적 검색)과 Rule-based(10대 지표 동적 가중치)가 결합된 하이브리드 추천 API 구현
- [x] 똥겜(MANIAC) 격리 로직 및 검색 결과 가뭄 방지(Fallback) UX 적용

## 🛠 진행 상황 및 핵심 기록
- **초기 데이터 파이프라인 구축 (The 5,000 Games Foundation)**
  - Steam API를 연동하여 평가가 어느 정도 보장된(혹은 숨겨진) 게임 데이터 **5,000개**를 1차 스크래핑 완료.
  - 게임의 제목, 설명, 장르 등의 텍스트 데이터를 GPT-4o-mini에 통과시켜 우리가 설계한 **14개 핵심 평가 지표(각 0~100점)**를 AI로 추론하여 수치화함.
  - OpenAI `text-embedding-3-small` 모델을 사용해 게임 설명을 벡터화하고, PostgreSQL(`pgvector`)에 5,000개의 데이터를 성공적으로 적재함.
- **관심사 분리 (Separation of Concerns)**
  - `data_worker.py`: 5,000개 데이터 스크래핑 + AI 분석 + 벡터 변환 및 DB 저장 역할로 격리.
  - `main.py`: 무거운 로직을 걷어내고 FastAPI 기반의 순수 추천/검색 서빙 API로 경량화.
- **하이브리드 추천 알고리즘 설계 (v2.2)**
  1. **의도 추출 (Intent Extraction)**: GPT를 이용해 유저 검색어에서 핵심 지표 추출 (예: `["story_depth", "art_style"]`).
  2. **1차 필터링**: Vector Similarity(코사인 유사도)를 활용해 DB에 구축된 5,000개 데이터 중 문맥이 맞는 Top 50 1차 싹쓸이.
  3. **동적 가중치 부스팅**: 유저가 요구한 지표의 가중치를 기본 1.5배로 시작하여, 기여도 2위권에 들도록 최대 1.7배(총합 2.55배)까지 동적 부스팅.
- **핵심 변수명 및 설정값 메모**
  - `DROP_THRESHOLD = 65`: 기본 추천 하한선 (이하 점수는 버림)
  - `DROP_THRESHOLD_FALLBACK = 60`: 결과가 3개 미만(`MIN_RESULTS_BEFORE_FALLBACK`)일 때 발동하는 심폐소생술 컷오프
  - `MAX_TOTAL_WEIGHT = 2.55`: 과적합을 막는 부스팅 하드웨어 락(Lock)
  - UX 강화: `boost_reason` 필드를 반환하여 프론트엔드에서 *"🚀 [스토리 깊이] 취향 집중 반영 (가중치 2.55배 폭발!)"* 형태의 설명 제공.

## 🚨 트러블슈팅 (문제 및 해결)
- **문제 1: 5,000개 데이터 스크래핑 및 AI 분석 중 API Rate Limit (속도 제한) 발생**
  - **원인:** 한 번에 너무 많은 게임 설명을 GPT-4o-mini와 Embedding API로 병렬 전송하여 OpenAI의 TPM/RPM 제한에 걸림.
  - **해결:** `data_worker.py`에 Chunk 단위(예: 100개씩) 처리 로직과 `asyncio.sleep()` 혹은 `tenacity` 라이브러리를 활용한 백오프(재시도) 로직을 추가하여 5,000개 데이터를 안전하게 적재 완료.
- **문제 2: 동적 가중치 부여 시 논리적 모순 및 무한 루프 위험**
  - **원인:** "유저가 요구한 지표를 무조건 Top 3 안에 넣는다"고 기획했으나, 유저가 4개 이상의 지표를 요구하면 수학적으로 성립하지 않음.
  - **해결:** 유저 요구 지표들(그룹 A)의 **'평균 기여도'**가 나머지 지표들의 2위 점수를 넘어서도록 보정하는 유니버설 룰 도입.
- **문제 3: DROP 컷오프(70점) 기준이 너무 높아 다양성 훼손**
  - **원인:** 컷오프를 70점으로 잡으니, 스토리는 별로지만 아트가 뛰어난 B급 명작들이 폐기(Drop)됨.
  - **해결:** 컷오프 하향(65점) 및 특정 S티어 지표만 높은 극단적 불균형 데이터는 `MANIAC(똥겜/극단적 취향)`으로 격리하여 프론트엔드 특설 코너용으로 구출.

## 💡 인사이트 및 다음 할 일
- **배운 점:** 양질의 데이터 5,000개를 먼저 구축해두고 하이브리드 추천 엔진(Vector RAG + Rule-based)을 물리리니, 검색 품질이 압도적으로 달라짐을 체감함. 도메인 지식(장르별 10대 지표 티어)과 수학적 안전장치(1.7배 Limit Cap, Fallback)의 결합이 UX를 어떻게 바꾸는지 배울 수 있었음.
- **다음 할 일:** - 프론트엔드(React/Next.js) 레포지토리 초기화 및 UI 프레임워크(Tailwind CSS) 세팅.
  - 백엔드 `/search` API를 연동하여 '숨겨진 보석 TOP 10' 카드 리스트와 '☠️ 극단적 취향 발굴단' 섹션 UI 구현.